# End-to-end product recommender

This notebook is the runnable walkthrough for the repository. It follows one path from Amazon Reviews JSONL through Spark preparation, training-only ID encoding, popularity baseline, two-tower training, temporal validation, a quality-gated serving bundle, and live FastAPI requests.

**Run from the repository root** in a Linux/WSL Jupyter kernel with Python 3.11, Java 21, and the packages in `pyproject.toml`. Spark on Windows/Python 3.14 has failed in this project. The first data run may download a large compressed dataset; set `MAX_RAW_ROWS` to a smaller sample for a quick walkthrough. The model run is intentionally bounded and is a plumbing demo, not a quality benchmark.

By default, experiment runs use local SQLite. To track runs in SageMaker, set `RECOMMENDER_MLFLOW_TRACKING_URI` to the ARN of the optional managed MLflow tracking server and install the `sagemaker-mlflow` extra. A model is registered only when its validation gate passes. The API's optional Bedrock explanation route is configured separately in Terraform.

All generated data and model outputs live under ignored `data/` and `artifacts/` paths. This notebook does not create AWS resources.

## Flow and source map

1. **Prepare interactions** — `src.data.prepare`: validate the JSONL fields, drop invalid/duplicate events, filter sparse users/items, sort each user's events by timestamp, and reserve the latest two events for validation and test.
2. **Encode IDs** — `src.models.prepare_tower_data`: fit stable user/item index mappings using training rows only. Held-out IDs absent from training are counted and excluded.
3. **Train** — `src.models.train_two_tower`: sample complete user histories within explicit bounds, draw unseen negative items, optimize pairwise BPR loss, and keep the checkpoint with the lowest sampled validation BPR loss.
4. **Evaluate** — `src.models.evaluate_two_tower`: rank a shared catalog after removing training-seen items; report Recall@K, Precision@K, and NDCG@K beside popularity on the same users.
5. **Gate and package** — `src.models.register_model` defines the validation gate; `src.pipeline.batch_job` applies it, records a serving policy, and writes checksummed model/vector/ID/history artifacts plus a popularity fallback.
6. **Optional experiment tracking and registration** — local SQLite is the default MLflow store. With a SageMaker tracking-server ARN configured, the notebook logs training and validation runs remotely and registers the model only if the validation gate passes.
7. **Serve** — `src.api.app`: verify the bundle, load it, serve recommendations, use popularity for unknown users or a failed two-tower gate, and expose health/model/metrics endpoints. The optional Bedrock route adds text after ranking without changing selected items.

The test split remains untouched here. Validation selects the serving policy; the final test should be evaluated once after model choices are frozen.

In [1]:
from pathlib import Path
import os
import sys
import platform
from datetime import datetime, timezone
import hashlib
import json
import urllib.request

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if not (ROOT / "pyproject.toml").is_file():
    raise RuntimeError("Open this notebook from within the project checkout.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")

print("Project:", ROOT)
print("Python:", platform.python_version(), platform.platform())
print("Kernel executable:", sys.executable)
print("Use Python 3.11 + Java 21 in WSL/Linux for the Spark steps.")

Project: C:\Users\aksha\PycharmProjects\EndtoEndMLProject
Python: 3.14.7 Windows-11-10.0.26300-SP0
Kernel executable: C:\Users\aksha\PycharmProjects\EndtoEndMLProject\venv\Scripts\python.exe
Use Python 3.11 + Java 21 in WSL/Linux for the Spark steps.


## 1. Configure a bounded run

The notebook reuses the prepared five-million-row dataset when present. Otherwise it downloads the compressed category file and prepares a 500,000-row prefix sample by default. Increase `MAX_RAW_ROWS` only when you have enough time, memory, and disk. A prefix sample may be biased by source ordering, so do not use its metrics as a final model claim.

In [ ]:
DATA_ROOT = ROOT / "data"
RAW_DIR = DATA_ROOT / "raw"
RAW_GZIP = RAW_DIR / "Movies_and_TV.jsonl.gz"
RAW_JSONL = RAW_DIR / "Movies_and_TV.jsonl"
PREFERRED_PREPARED = DATA_ROOT / "processed" / "movies_tv_5m"
NOTEBOOK_PREPARED = DATA_ROOT / "processed" / "notebook_movies_tv"
ENCODED_DIR = NOTEBOOK_PREPARED / "two_tower" / "v1"

MAX_RAW_ROWS = 500_000
MIN_USER_INTERACTIONS = 5
MIN_ITEM_INTERACTIONS = 5
MAX_USERS = 500
MAX_TRAIN_PAIRS = 30_000
EPOCHS = 2
SEED = 42
K_VALUES = [10, 20]

if (PREFERRED_PREPARED / "train").is_dir():
    PREPARED_DIR = PREFERRED_PREPARED
    print("Reusing:", PREPARED_DIR)
elif (NOTEBOOK_PREPARED / "train").is_dir():
    PREPARED_DIR = NOTEBOOK_PREPARED
    print("Reusing:", PREPARED_DIR)
else:
    PREPARED_DIR = NOTEBOOK_PREPARED
    print("Will prepare a fresh bounded dataset at:", PREPARED_DIR)

## 2. Start Spark and prepare the temporal data

Spark uses lazy DataFrame transformations: selecting, filtering, joining, and windowing define the work; counts and Parquet writes execute it. The latest interaction is test, the previous is validation, and earlier interactions are training. The preparation implementation filters sparse entities before splitting, so this first project version has an eligibility leakage limitation documented in the study guide.

In [ ]:
from pyspark.sql import SparkSession
from pyspark import StorageLevel
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("recommender-end-to-end-notebook")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

In [ ]:
from src.data.prepare import load_reviews, filter_interactions, write_dataset

if not (PREPARED_DIR / "train").is_dir():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    raw_path = RAW_JSONL if RAW_JSONL.is_file() else RAW_GZIP
    if not raw_path.is_file():
        url = "https://mcauleylab.ucsd.edu/public_datasets/data/amazon_2023/raw/review_categories/Movies_and_TV.jsonl.gz"
        print("Downloading dataset to", RAW_GZIP)
        urllib.request.urlretrieve(url, RAW_GZIP)
        raw_path = RAW_GZIP

    reviews = load_reviews(spark, raw_path, MAX_RAW_ROWS).persist(StorageLevel.MEMORY_AND_DISK)
    interactions = filter_interactions(reviews, MIN_USER_INTERACTIONS, MIN_ITEM_INTERACTIONS)
    print("Filtered interaction rows:", interactions.count())
    write_dataset(interactions, PREPARED_DIR)
    reviews.unpersist(blocking=True)

train_raw = spark.read.parquet(str(PREPARED_DIR / "train"))
validation_raw = spark.read.parquet(str(PREPARED_DIR / "validation"))
test_raw = spark.read.parquet(str(PREPARED_DIR / "test"))
print("Prepared rows:", {
    "train": train_raw.count(),
    "validation": validation_raw.count(),
    "test": test_raw.count(),
})
train_raw.printSchema()
train_raw.show(5, truncate=True)

## 3. Build training-only ID mappings

Embedding tables need contiguous integer IDs. We fit the vocabularies only on the training split, then inner-join validation and test through those maps. The printed excluded-row counts are important: they show how much held-out activity belongs to unseen users/items and therefore cannot be scored by this ID-only model.

In [ ]:
from src.models.prepare_tower_data import build_id_mapping, encode_split

user_mapping = build_id_mapping(train_raw, "user_id", "user_idx")
item_mapping = build_id_mapping(train_raw, "item_id", "item_idx")
ENCODED_DIR.mkdir(parents=True, exist_ok=True)
user_mapping.write.mode("overwrite").parquet(str(ENCODED_DIR / "user_mapping"))
item_mapping.write.mode("overwrite").parquet(str(ENCODED_DIR / "item_mapping"))

encoded_counts = {}
for split_name, source in [
    ("train", train_raw),
    ("validation", validation_raw),
    ("test", test_raw),
]:
    encoded, excluded = encode_split(source, user_mapping, item_mapping)
    encoded.write.mode("overwrite").parquet(str(ENCODED_DIR / split_name))
    encoded_counts[split_name] = {"encoded_rows": encoded.count(), "excluded_oov_rows": excluded}

print(encoded_counts)
print("Users:", user_mapping.count(), "Items:", item_mapping.count())

## 4. Sample training rows and fit the two-tower model

Each positive interaction is paired with an item the user has not seen in training. The model learns user and item embeddings; a dot product scores a pair. BPR loss encourages observed items to outrank sampled negatives. This bounded sample intentionally mirrors the batch smoke configuration and may be too small to beat popularity.

In [ ]:
from src.models.train_two_tower import (
    TrainingConfig, _read_sampled_pairs, train_two_tower
)

train_pairs, validation_pairs, model_user_ids, num_items = _read_sampled_pairs(
    spark,
    ENCODED_DIR,
    max_users=MAX_USERS,
    max_train_pairs=MAX_TRAIN_PAIRS,
    seed=SEED,
)
spark.stop()

print({
    "users_selected": len(model_user_ids),
    "train_pairs": len(train_pairs),
    "validation_pairs": len(validation_pairs),
    "catalog_items": num_items,
})
assert train_pairs and validation_pairs

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = ROOT / "artifacts" / "notebook_runs" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
CHECKPOINT = RUN_DIR / "best_model.pt"

config = TrainingConfig(
    embedding_dim=16,
    batch_size=512,
    epochs=EPOCHS,
    learning_rate=0.001,
    weight_decay=0.0001,
    seed=SEED,
    device="cpu",
)
training = train_two_tower(
    train_pairs,
    validation_pairs,
    num_users=len(model_user_ids),
    num_items=num_items,
    config=config,
    model_user_ids=model_user_ids,
    item_mapping_path=str(ENCODED_DIR / "item_mapping"),
    checkpoint_path=CHECKPOINT,
)
(RUN_DIR / "history.json").write_text(
    json.dumps({
        "best_epoch": training.best_epoch,
        "best_validation_loss": training.best_validation_loss,
        "epochs": training.history,
    }, indent=2)
)
print("Best epoch:", training.best_epoch)
print("Best validation pairwise loss:", round(training.best_validation_loss, 6))
print("Checkpoint:", CHECKPOINT)

## 5. Evaluate on validation and compare with popularity

Recall@K is the share of held-out relevant items retrieved in the top K. NDCG@K also rewards placing a hit higher. There is one validation target per user here, so Precision@K is Recall@K divided by K. Popularity is computed only from training interactions and evaluated on the same users and catalog.

In [ ]:
from src.models.evaluate_two_tower import evaluate_validation

validation_report = evaluate_validation(
    checkpoint_path=CHECKPOINT,
    data_path=ENCODED_DIR,
    ks=K_VALUES,
    split="validation",
)
(RUN_DIR / "validation_metrics.json").write_text(json.dumps(validation_report, indent=2))

comparison_rows = []
for method, at_k in validation_report["metrics_at_k"].items():
    for k, metric in at_k.items():
        comparison_rows.append({
            "model": method,
            "K": int(k),
            "recall": metric["recall"],
            "ndcg": metric["ndcg"],
            "users": metric["users"],
        })

import pandas as pd
display(pd.DataFrame(comparison_rows).sort_values(["K", "model"]).reset_index(drop=True))

## 6. Apply the release gate and choose a serving policy

The project's gate requires at least a 5% NDCG@10 lift over popularity and no Recall@10 regression. A failed candidate does not get mislabeled as a successful learned model: the batch bundle uses popularity. The same popularity ranking is also the cold-start fallback for unknown users.

In [ ]:
from src.models.register_model import check_validation_gate

try:
    gate = {"passed": True, **check_validation_gate(validation_report)}
except ValueError as error:
    gate = {"passed": False, "reason": str(error)}

SERVING_POLICY = "two_tower" if gate["passed"] else "popularity"
print({"validation_gate": gate, "serving_policy": SERVING_POLICY})

## 6.1 Optional SageMaker MLflow tracking and model registry

The notebook logs the training history and validation metrics to the configured MLflow tracking URI. The default is local SQLite. For SageMaker, set `RECOMMENDER_MLFLOW_TRACKING_URI` to the tracking server ARN and install `uv sync --extra sagemaker-mlflow`. The Terraform option enables automatic synchronization of registered MLflow models into SageMaker Model Registry. Registration is attempted only after the validation gate passes; the held-out test split is not used.

In [ ]:
import os
from src.models.experiment_tracking import log_experiment_run
from src.models.register_model import register_checkpoint

TRACKING_URI = os.getenv("RECOMMENDER_MLFLOW_TRACKING_URI", "sqlite:///mlflow.db")
if TRACKING_URI.startswith("arn:aws:sagemaker:"):
    try:
        import sagemaker_mlflow  # noqa: F401 — registers SageMaker's SigV4 MLflow plugin.
    except ImportError as error:
        raise RuntimeError(
            "Install the optional dependency with `uv sync --extra sagemaker-mlflow`."
        ) from error

training_mlflow_run_id = log_experiment_run(
    experiment_name="two-tower-recommender",
    run_name=f"training-{RUN_ID}",
    parameters={
        "embedding_dim": config.embedding_dim,
        "batch_size": config.batch_size,
        "epochs": config.epochs,
        "learning_rate": config.learning_rate,
        "weight_decay": config.weight_decay,
        "seed": SEED,
        "max_users": MAX_USERS,
        "max_train_pairs": MAX_TRAIN_PAIRS,
    },
    metrics={
        "best_validation_bpr_loss": float(training.best_validation_loss),
        "best_epoch": float(training.best_epoch),
    },
    artifacts=[RUN_DIR / "history.json"],
    step_metrics=[
        {
            "epoch": int(row["epoch"]),
            "train_bpr_loss": float(row["train_loss"]),
            "validation_bpr_loss": float(row["validation_loss"]),
        }
        for row in training.history
    ],
    tags={"pipeline_run_id": RUN_ID, "split": "train_and_validation"},
    tracking_uri=TRACKING_URI,
)

evaluation_metrics = {}
for method, by_k in validation_report["metrics_at_k"].items():
    metric_prefix = method.replace("-", "_")
    for k, values in by_k.items():
        for metric_name in ("recall", "precision", "ndcg"):
            evaluation_metrics[f"{metric_prefix}_{metric_name}_at_{k}"] = float(values[metric_name])

evaluation_mlflow_run_id = log_experiment_run(
    experiment_name="two-tower-recommender",
    run_name=f"validation-{RUN_ID}",
    parameters={"seed": SEED, "ks": K_VALUES, "split": "validation"},
    metrics=evaluation_metrics,
    artifacts=[RUN_DIR / "validation_metrics.json"],
    tags={
        "pipeline_run_id": RUN_ID,
        "validation_gate_passed": str(bool(gate["passed"])).lower(),
        "serving_policy": SERVING_POLICY,
    },
    tracking_uri=TRACKING_URI,
)

registration_result = None
if gate["passed"]:
    registration_result = register_checkpoint(
        checkpoint=CHECKPOINT,
        validation_report=RUN_DIR / "validation_metrics.json",
        registered_name="amazon-reviews-two-tower",
        tracking_uri=TRACKING_URI,
    )
    print("Registered validation-approved model:", registration_result)
else:
    print("Model Registry unchanged: the validation gate did not pass.")

print({
    "tracking_uri": TRACKING_URI,
    "training_mlflow_run_id": training_mlflow_run_id,
    "evaluation_mlflow_run_id": evaluation_mlflow_run_id,
})

## 7. Export and inspect the serving bundle

The bundle contains the checkpoint, item vectors and original item IDs, model user IDs, training-seen histories, a popularity ranking, model metadata, and SHA-256 checksums. The API verifies those checksums during startup. Run IDs and an input path are included as lineage; this notebook uses a direct in-kernel training flow rather than the MLflow CLI wrapper.

In [ ]:
from src.pipeline.batch_job import export_serving_artifacts

spark = (
    SparkSession.builder.appName("recommender-notebook-popularity")
    .master("local[*]")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
popularity_rows = (
    spark.read.parquet(str(ENCODED_DIR / "train"))
    .groupBy("item_idx").count()
    .orderBy("count", "item_idx", ascending=[False, True])
    .select("item_idx").collect()
)
popularity_indices = [int(row.item_idx) for row in popularity_rows]
spark.stop()

model_version = "sha256:" + hashlib.sha256(CHECKPOINT.read_bytes()).hexdigest()
lineage = {
    "model_version": model_version,
    "pipeline_run_id": RUN_ID,
    "training_mlflow_run_id": training_mlflow_run_id,
    "evaluation_mlflow_run_id": evaluation_mlflow_run_id,
    "trained_at_utc": datetime.now(timezone.utc).isoformat(),
    "seed": SEED,
    "serving_policy": SERVING_POLICY,
}
SERVING_DIR = RUN_DIR / "serving"
bundle_path = export_serving_artifacts(
    ENCODED_DIR,
    SERVING_DIR,
    CHECKPOINT,
    lineage=lineage,
    popularity_item_indices=popularity_indices,
)
print("Bundle:", bundle_path, "bytes:", bundle_path.stat().st_size)
print("Serving manifest:", json.loads((SERVING_DIR / "manifest.json").read_text()))

## 8. Run the API in-process and inspect responses

This avoids starting a separate server process, so the notebook stays self-contained. The same FastAPI app is used by the container. Test a known mapped user, an unknown user (popularity fallback), readiness, model lineage, and Prometheus metrics.

In [ ]:
from fastapi.testclient import TestClient
from src.api.app import create_app

known_user = model_user_ids[0]
app = create_app(SERVING_DIR)
with TestClient(app) as client:
    ready_response = client.get("/ready")
    known_response = client.get(f"/recommendations/{known_user}?limit=10")
    unknown_response = client.get("/recommendations/999999999?limit=10")
    info_response = client.get("/model-info")
    metrics_response = client.get("/metrics")

print("ready:", ready_response.status_code, ready_response.json())
print("known user:", known_response.status_code, known_response.json())
print("unknown user fallback:", unknown_response.status_code, unknown_response.json())
print("model info:", info_response.status_code, info_response.json())
print("metrics endpoint:", metrics_response.status_code,
      "request metric present:", b"recommendation_requests_total" in metrics_response.content)
assert ready_response.status_code == 200
assert known_response.status_code == 200
assert unknown_response.status_code == 200

## 9. Results, limitations, and optional container run

A successful run proves that data can flow through training, evaluation, artifact export, and API serving. It does **not** prove large-scale training, online business lift, cloud deployment, or production SLOs. The 500-user/two-epoch defaults are deliberately a smoke run. For a heavier experiment, increase `MAX_RAW_ROWS`, `MAX_USERS`, `MAX_TRAIN_PAIRS`, and `EPOCHS` one at a time, keep validation for selection, and reserve test for a final frozen comparison.

To run the same bundle in Docker Compose, set `MODEL_DIR` to `SERVING_DIR` in a terminal and run `docker compose up --build`; this starts the API, Redis, Prometheus, and OpenTelemetry Collector. The local Jupyter flow above exercises the API directly without shelling out.

In [ ]:
print({
    "prepared_data": str(PREPARED_DIR),
    "encoded_data": str(ENCODED_DIR),
    "run_dir": str(RUN_DIR),
    "model_version": model_version,
    "serving_policy": SERVING_POLICY,
    "validation_at_k": validation_report["metrics_at_k"],
    "bundle": str(bundle_path),
})

## 10. Read implementation code from this notebook

The walkthrough above explains the call order and executes the real project functions. Use this final section when you want to inspect the implementation body without navigating away from the notebook. Choose a function name and run the cell; source appears inline. The test notebook provides the parallel map from behavior to test files.

In [3]:
import inspect
from IPython.display import display, Markdown, Code

from src.data import prepare as data_prepare
from src.models import prepare_tower_data as id_prepare
from src.models import train_two_tower as tower_training
from src.models import evaluate_two_tower as ranking_evaluation
from src.models import register_model as model_gate
from src.models import popularity as popularity_baseline
from src.models import two_tower as tower_model
from src.pipeline import batch_job as batch_pipeline
from src.api import app as serving_api

IMPLEMENTATION_MAP = {
    "load_reviews": data_prepare.load_reviews,
    "filter_interactions": data_prepare.filter_interactions,
    "rank_interactions": data_prepare.rank_interactions,
    "split_ranked": data_prepare.split_ranked,
    "build_id_mapping": id_prepare.build_id_mapping,
    "encode_split": id_prepare.encode_split,
    "negative_sampling": tower_training._fixed_validation_dataset,
    "train_two_tower": tower_training.train_two_tower,
    "TwoTowerRecommender": tower_model.TwoTowerRecommender,
    "recommend_top_k": ranking_evaluation.recommend_top_k,
    "ranking_metrics_at_k": ranking_evaluation.ranking_metrics_at_k,
    "evaluate_validation": ranking_evaluation.evaluate_validation,
    "popularity_baseline": popularity_baseline.rank_popular_items,
    "validation_gate": model_gate.check_validation_gate,
    "export_serving_artifacts": batch_pipeline.export_serving_artifacts,
    "load_runtime": serving_api.load_runtime,
    "create_app": serving_api.create_app,
}

print("Available implementation names:")
print(", ".join(IMPLEMENTATION_MAP))
# Example:
# display(Code(inspect.getsource(IMPLEMENTATION_MAP["train_two_tower"]), language="python"))

Available implementation names:
load_reviews, filter_interactions, rank_interactions, split_ranked, build_id_mapping, encode_split, negative_sampling, train_two_tower, TwoTowerRecommender, recommend_top_k, ranking_metrics_at_k, evaluate_validation, popularity_baseline, validation_gate, export_serving_artifacts, load_runtime, create_app


The implementation browser is intentionally on demand so running the whole notebook does not dump large source listings into its outputs. Example: set `FUNCTION_TO_INSPECT = "recommend_top_k"`, then run the next cell.

In [ ]:
FUNCTION_TO_INSPECT = "recommend_top_k"
if FUNCTION_TO_INSPECT not in IMPLEMENTATION_MAP:
    raise KeyError(f"Choose one of: {', '.join(IMPLEMENTATION_MAP)}")
display(Code(inspect.getsource(IMPLEMENTATION_MAP[FUNCTION_TO_INSPECT]), language="python"))